<p align="center">
<img src="Images/sorbonne_logo.png" alt="Logo" width="300"/>
</p>

# **Module 2 - Statistics & Data Transformation**

* **Author**: Elia Landini
* **Student ID**: 12310239
* **Course**: EESM2-Financial Economics 
* **Supervisor**: Jean-Bernard Chatelain
* **Reference Repository**: https://github.com/EliaLand/Policy_Target_RegimeSwitchingPersistence

### **1) REQUIREMENTS SET-UP**

In [119]:
# Requirements.txt file installation
# !pip install -r requirements.txt

In [120]:
# Libraries import
import warnings
import pandas as pd
import numpy as np
import random
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.colors import Normalize
import matplotlib.dates as mdates
%matplotlib inline
import seaborn as sns
import scipy.stats as stats
from scipy.stats import norm
from scipy.stats import levene
from scipy.stats import ks_2samp
from scipy.stats import kstest
from scipy.stats import pearsonr
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.filters.hp_filter import hpfilter
import sklearn.tree
import sklearn.metrics
import sklearn.metrics
import sklearn.model_selection
import sklearn.preprocessing 
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (roc_auc_score, roc_curve, confusion_matrix,
                             precision_score, recall_score, f1_score,
                             accuracy_score, precision_recall_curve, auc, 
                             RocCurveDisplay, ConfusionMatrixDisplay)
from sklearn.linear_model import (LinearRegression, LogisticRegression)
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.utils.class_weight import compute_class_weight
import plotly.express as px
import openpyxl as pxl
from stargazer.stargazer import Stargazer
from IPython.core.display import HTML
from IPython.display import Image
import itertools
from arch.unitroot import PhillipsPerron

### **2) DESCRIPTIVE STATISTICS (RAW)**

In [121]:
# df import
jp_aggregated_df = pd.read_csv("Data/Aggregated/jp_aggregated_df.csv")
jp_aggregated_df

,Country,Time,Monetary Aggregates - M1 (JPY),Monetary Aggregates - M2 (JPY),Monetary Aggregates - M3 (JPY),Total Credit - Private Non-Financial (%GDP),Total Credit - General Government (%GDP),Total Credit - Households & NPISHs (%GDP),Total Treasury Reserves (- Gold),10-Year Gov Bond Yields (%),...,Real GDP (billions chained 2015 JPY),Central Government Debt (% GDP),Domestic Private Debt Securities (% GDP),Domestic Public Debt Securities (% GDP),BoJ’s Total Assets (100 Million Yen),Loan Interest Rate (%),Deposit Interest Rate (%),1615.T-Price,10-Year US T-Bills Yield (%),CBOE-VIX
0,JP,1950-12,NaN,NaN,NaN,NaN,NaN,NaN,5.980000e+02,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,JP,1951-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,JP,1951-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,JP,1951-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,JP,1951-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
899,JP,2025-11,NaN,NaN,NaN,NaN,NaN,NaN,1.245364e+06,1.805,...,591856.7,NaN,NaN,NaN,6979490.0,1.139,0.225,505.500000,4.093889,19.769500
900,JP,2025-12,NaN,NaN,NaN,NaN,NaN,NaN,1.252603e+06,2.060,...,591856.7,NaN,NaN,NaN,6777762.0,1.404,0.225,528.200012,4.143182,15.548182
901,JP,2026-01,NaN,NaN,NaN,NaN,NaN,NaN,1.259248e+06,2.240,...,NaN,NaN,NaN,NaN,6828680.0,NaN,0.227,598.299988,4.213500,16.179048
902,JP,2026-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,6837705.0,NaN,0.296,647.900024,4.125789,19.207000


In [122]:
# Metrics clusters for plotting
metrics_group = {
    "Monetary Aggregates" : ["Monetary Aggregates - M1 (JPY)", "Monetary Aggregates - M2 (JPY)", "Monetary Aggregates - M3 (JPY)"],
    "Credit Metrics" : ["Total Credit - General Government (%GDP)", "Total Credit - Households & NPISHs (%GDP)", "Total Credit - Private Non-Financial (%GDP)"],
    "Reserves" : ["Total Treasury Reserves (- Gold)"],
    "Monetary Policy Proxies (Yields)" : ["10-Year Gov Bond Yields (%)", "Call Money/Interbank Immediate (%)", "Est. 1-year Neutral Interest Rate (%)", "Est. 10-year Neutral Interest Rate (%)"], 
    "Exchange Rate" : ["USD-JPY reer CPI-based (Index 2015=100)", "JPY-USD Spot Exchange Rate"],
    "Output-Trends": ["Real GDP (billions chained 2015 JPY)"],
    "Consumption Prices": ["HICP (NSA)"],
    "Debt Metrics" : ["Central Government Debt (% GDP)", "Domestic Private Debt Securities (% GDP)", "Domestic Public Debt Securities (% GDP)"],
    "Banking Sector" : ["Loan Interest Rate (%)", "Deposit Interest Rate (%)", "1615.T-Price"],
    "Controls" : ["10-Year US T-Bills Yield (%)", "CBOE-VIX"],
    "BoJ Total Assets" : ["BoJ’s Total Assets (100 Million Yen)"]
}

In [123]:
# Descriptive statistics summary table - aggregate data
df = jp_aggregated_df.copy()
df.describe()

,Monetary Aggregates - M1 (JPY),Monetary Aggregates - M2 (JPY),Monetary Aggregates - M3 (JPY),Total Credit - Private Non-Financial (%GDP),Total Credit - General Government (%GDP),Total Credit - Households & NPISHs (%GDP),Total Treasury Reserves (- Gold),10-Year Gov Bond Yields (%),Call Money/Interbank Immediate (%),Est. 1-year Neutral Interest Rate (%),...,Real GDP (billions chained 2015 JPY),Central Government Debt (% GDP),Domestic Private Debt Securities (% GDP),Domestic Public Debt Securities (% GDP),BoJ’s Total Assets (100 Million Yen),Loan Interest Rate (%),Deposit Interest Rate (%),1615.T-Price,10-Year US T-Bills Yield (%),CBOE-VIX
count,7.670000e+02,7.460000e+02,5.270000e+02,729.000000,333.000000,729.000000,8.360000e+02,445.000000,487.000000,372.000000,...,384.000000,385.000000,277.000000,277.000000,3.350000e+02,387.000000,489.000000,218.000000,771.000000,435.000000
mean,2.799125e+14,3.659118e+14,9.357442e+14,163.508642,175.399099,54.214815,4.140074e+05,1.810333,1.247216,0.353968,...,535407.262500,129.955192,61.365237,140.770820,3.238492e+06,1.366514,1.347229,157.056886,5.816945,19.463917
std,2.845290e+14,3.038362e+14,3.323003e+14,27.939645,41.642344,15.385155,5.108432e+05,1.825189,2.185570,0.580089,...,36655.682733,57.276103,8.777222,37.063681,2.517772e+06,0.687633,2.239623,99.039047,2.937092,7.396644
min,4.021870e+12,3.564500e+12,2.952000e+14,110.800000,90.100000,19.900000,5.980000e+02,-0.280000,-0.071000,-0.420000,...,461491.900000,38.176187,46.447950,63.163390,6.800640e+05,0.465000,0.003250,67.772186,0.623636,10.125455
25%,5.802214e+13,4.820300e+13,7.623000e+14,139.100000,143.600000,44.100000,1.330404e+04,0.566000,0.001000,-0.260500,...,503947.575000,80.478284,55.154430,111.204600,1.186694e+06,0.789000,0.029000,103.542906,3.888357,14.104177
50%,1.518072e+14,3.102086e+14,9.926000e+14,162.100000,183.600000,61.000000,8.137040e+04,1.333000,0.091000,0.243500,...,535754.100000,128.599668,59.193390,157.152800,1.505173e+06,1.309000,0.120000,126.242756,5.460000,17.683182
75%,4.826217e+14,6.484915e+14,1.131859e+15,183.700000,211.500000,66.900000,9.825051e+05,1.948000,0.804870,0.864250,...,568025.350000,192.112850,68.579800,176.609800,5.594752e+06,1.696000,0.956000,155.226978,7.527208,23.014426
max,1.081546e+15,9.632187e+14,1.597004e+15,214.200000,237.700000,71.100000,1.371116e+06,8.032000,8.278130,1.380000,...,593799.900000,216.135005,78.846620,217.026000,7.648115e+06,4.043000,8.347500,647.900024,15.323810,62.668947


### **3) STATIONARITY TESTING**

In [124]:
# Autocorrelation coefficients AR(1)
# Drop non-numeric columns and rows with missing values
df = jp_aggregated_df.copy()
jp_aggregated_numeric = df.drop(columns=["Country", "Time"]).dropna()

# AR(1) autocorrelation for each variable
ar1_results = {}
for col in jp_aggregated_numeric.columns:
    series = jp_aggregated_numeric[col]

# (!!!) lag-1 autocorrelation
    ar1 = series.autocorr(lag=1)
    ar1_results[col] = ar1

# Better to create a dataframe to display the results
jp_ar1_df = pd.DataFrame.from_dict(ar1_results, orient="index", columns=["AR(1)"])
jp_ar1_df

,AR(1)
Monetary Aggregates - M1 (JPY),0.999761
Monetary Aggregates - M2 (JPY),0.999878
Monetary Aggregates - M3 (JPY),0.999875
Total Credit - Private Non-Financial (%GDP),0.982539
Total Credit - General Government (%GDP),0.997181
Total Credit - Households & NPISHs (%GDP),0.975292
Total Treasury Reserves (- Gold),0.990202
10-Year Gov Bond Yields (%),0.983519
Call Money/Interbank Immediate (%),0.985473
Est. 1-year Neutral Interest Rate (%),0.985248


In [125]:
# Unit-root Testing - Adfuller Test 
# Drop non-numeric columns and handle missing data
df = jp_aggregated_df.copy()
jp_aggregated_numeric = df.drop(columns=["Country", "Time"]).dropna()

# (!!!) We need to initialize the results as empty list before execuding the test
results = []

for col in jp_aggregated_numeric.columns:
    series = jp_aggregated_numeric[col]

# As before, we extract the AR(1) coefficients
    ar1 = series.autocorr(lag=1)

# Augmented Dickey-Fuller (ADF) unit root test 
    adf_result = adfuller(series, autolag="AIC")
    adf_stat = adf_result[0]
    p_value = adf_result[1]
    crit_values = adf_result[4]

    results.append({
        "Variable": col,
        "AR(1)": ar1,
        "ADF Statistic": adf_stat,
        "p-value": p_value,
        "Stationary - Absence of unit-root (HP1)": "Yes" if p_value < 0.05 else "No"
    })

jp_adf_df = pd.DataFrame(results)
jp_adf_df

,Variable,AR(1),ADF Statistic,p-value,Stationary - Absence of unit-root (HP1)
0,Monetary Aggregates - M1 (JPY),0.999761,2.145042,0.998834,No
1,Monetary Aggregates - M2 (JPY),0.999878,7.421314,1.000000,No
2,Monetary Aggregates - M3 (JPY),0.999875,6.495981,1.000000,No
3,Total Credit - Private Non-Financial (%GDP),0.982539,-1.418998,0.573135,No
4,Total Credit - General Government (%GDP),0.997181,-3.315840,0.014185,Yes
5,Total Credit - Households & NPISHs (%GDP),0.975292,-1.371612,0.595807,No
6,Total Treasury Reserves (- Gold),0.990202,-1.598362,0.484416,No
7,10-Year Gov Bond Yields (%),0.983519,-0.734868,0.837510,No
8,Call Money/Interbank Immediate (%),0.985473,-3.532602,0.007188,Yes
9,Est. 1-year Neutral Interest Rate (%),0.985248,-1.543562,0.511863,No


In [126]:
# Unit-root Testing - Phillips-Perron Test 
# (!!!) We need to initialize the results as empty list before execuding the test
pp_results = []

for col in jp_aggregated_numeric.columns:
    series = jp_aggregated_numeric[col].dropna()
    
# Phillips–Perron test 
# (!!!) From arch instead of stats.models is much smoother
    test = PhillipsPerron(series)
    pp_results.append({
        "Variable": col,
        "PP Statistic": test.stat,
        "p-value": test.pvalue,
        "Stationary - Absence of unit-root (HP1)": "Yes" if test.pvalue < 0.05 else "No"
    })

jp_pp_df = pd.DataFrame(pp_results)
jp_pp_df

,Variable,PP Statistic,p-value,Stationary - Absence of unit-root (HP1)
0,Monetary Aggregates - M1 (JPY),5.499319,1.000000,No
1,Monetary Aggregates - M2 (JPY),15.311005,1.000000,No
2,Monetary Aggregates - M3 (JPY),11.710651,1.000000,No
3,Total Credit - Private Non-Financial (%GDP),-1.306735,0.626085,No
4,Total Credit - General Government (%GDP),-2.363247,0.152379,No
5,Total Credit - Households & NPISHs (%GDP),-1.806462,0.377258,No
6,Total Treasury Reserves (- Gold),-1.615145,0.475275,No
7,10-Year Gov Bond Yields (%),-0.282190,0.927989,No
8,Call Money/Interbank Immediate (%),-2.927652,0.042226,Yes
9,Est. 1-year Neutral Interest Rate (%),-1.479774,0.543492,No


### **4) NON-STATIONARITY CORRECTIONS**

In [127]:
# Non-Stationarity Corrections - Log-Difference Transformations
# (!!!) Basically we need a detrending transformation for all the variables as expected, given the undisputable presence of unit-root root and so non-sttaionarity
# (!!!) Autocorrelation is also evident, suggesting a marked time-dependent component, and so a trend, so we'll opt for both log transformations as well as first differences 
df = jp_aggregated_df.copy()
trans_df = df[["Country", "Time"]].copy()

# Transformations: 
# - Monetary Aggregates: I(1) nominal levels (levels non-stationary, but first-differences are I(0), stationary)
# - Reserves: I(1), level series of policy shocks 
# - Exchange Rate: Log-difference (returns)
# - Consumption Prices: Log-difference (inflation)
# - Banking metruics: Log-difference (returns)
# - BoJ’s Total Assets: CA smoothing 
log_transformed_variables = ["Monetary Aggregates - M1 (JPY)", "Monetary Aggregates - M2 (JPY)", "Monetary Aggregates - M3 (JPY)",
                             "Total Treasury Reserves (- Gold)",
                             "USD-JPY reer CPI-based (Index 2015=100)", "JPY-USD Spot Exchange Rate",
                             "HICP (SA)",
                             "1615.T-Price",
                             "BoJ’s Total Assets (100 Million Yen)"
                            ]

# Log-Difference Transformed Variables and Annualization (percentage change over 12 months)
for var in log_transformed_variables:
    if var != "HICP (SA)":
        trans_df[f"LogDiff-{var}"] = np.log(df[f"{var}"]).diff()
    else:
        trans_df[f"LogDiff-{var}"] = np.log(df[f"{var}"]).diff()*12*100
trans_df


,Country,Time,LogDiff-Monetary Aggregates - M1 (JPY),LogDiff-Monetary Aggregates - M2 (JPY),LogDiff-Monetary Aggregates - M3 (JPY),LogDiff-Total Treasury Reserves (- Gold),LogDiff-USD-JPY reer CPI-based (Index 2015=100),LogDiff-JPY-USD Spot Exchange Rate,LogDiff-HICP (SA),LogDiff-1615.T-Price,LogDiff-BoJ’s Total Assets (100 Million Yen)
0,JP,1950-12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,JP,1951-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,JP,1951-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,JP,1951-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,JP,1951-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
899,JP,2025-11,NaN,NaN,NaN,0.005714,-0.021022,0.024710,NaN,0.076439,0.002295
900,JP,2025-12,NaN,NaN,NaN,0.005796,-0.013176,0.004976,NaN,0.043927,-0.029329
901,JP,2026-01,NaN,NaN,NaN,0.005291,-0.013183,0.004706,NaN,0.124617,0.007484
902,JP,2026-02,NaN,NaN,NaN,NaN,NaN,-0.009937,NaN,0.079644,0.001321


In [128]:
# Non-Stationarity Corrections - AR(1) detrending 
df = jp_aggregated_df.copy()
trans_df = trans_df.copy()

# Transformations: 
# - Credit Metrics: AR(1) detrending (cyclical credit gap), we want to remove the persistence of credit t-1, we can reduce this relatiosnship to:
# (!!!) c_t = α + ρc_t−1​ + ε_t, we take the residuals 
# (!!!) first differences are to aggresive for credit metrics, they destroy medium-term cycles, signal-to-noise
# (!!!) credit metrics tend to be highly-persistent I(0) and near-unit-root stationary (so it might result non-stationary, but it is actually slowly mean reverting, we want to keep cyclical deviations)
# - Monetary Policy Proxies: AR(1) detrending (policy shocks), rates are persistent but not truly I(1)
# - Debt Metrics: same reasoning as for credit 
# - Banking metrics: same reasoning as for yields
# - VIX index
ar1detrend_transformed_variables = ["Total Credit - General Government (%GDP)", "Total Credit - Households & NPISHs (%GDP)", "Total Credit - Private Non-Financial (%GDP)",
                                    "10-Year Gov Bond Yields (%)", "Call Money/Interbank Immediate (%)", "Est. 1-year Neutral Interest Rate (%)", "Est. 10-year Neutral Interest Rate (%)",
                                    "Central Government Debt (% GDP)", "Domestic Private Debt Securities (% GDP)", "Domestic Public Debt Securities (% GDP)",
                                    "Loan Interest Rate (%)", "Deposit Interest Rate (%)",
                                    "10-Year US T-Bills Yield (%)", "CBOE-VIX"
                                   ]

# AR(1) detrending Transformed Variables 
for var in ar1detrend_transformed_variables:
# Lag-1 Reg
# (!!!) It triggers an errors as we don't specificy the model time index and frequency. It should be harmless, it only deactivates forecasting features.
    model = AutoReg(df[f"{var}"].dropna(), lags=1, old_names=False).fit()
# Residuals 
    trans_df[f"AR(1)detrend-{var}"] = df[f"{var}"] - model.fittedvalues
trans_df

d:\Conda\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
d:\Conda\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
d:\Conda\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
d:\Conda\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use

,Country,Time,LogDiff-Monetary Aggregates - M1 (JPY),LogDiff-Monetary Aggregates - M2 (JPY),LogDiff-Monetary Aggregates - M3 (JPY),LogDiff-Total Treasury Reserves (- Gold),LogDiff-USD-JPY reer CPI-based (Index 2015=100),LogDiff-JPY-USD Spot Exchange Rate,LogDiff-HICP (SA),LogDiff-1615.T-Price,...,AR(1)detrend-Call Money/Interbank Immediate (%),AR(1)detrend-Est. 1-year Neutral Interest Rate (%),AR(1)detrend-Est. 10-year Neutral Interest Rate (%),AR(1)detrend-Central Government Debt (% GDP),AR(1)detrend-Domestic Private Debt Securities (% GDP),AR(1)detrend-Domestic Public Debt Securities (% GDP),AR(1)detrend-Loan Interest Rate (%),AR(1)detrend-Deposit Interest Rate (%),AR(1)detrend-10-Year US T-Bills Yield (%),AR(1)detrend-CBOE-VIX
0,JP,1950-12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,JP,1951-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,JP,1951-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,JP,1951-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,JP,1951-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
899,JP,2025-11,NaN,NaN,NaN,0.005714,-0.021022,0.024710,NaN,0.076439,...,0.006290,0.002655,0.002863,NaN,NaN,NaN,-0.129205,0.004569,0.024252,1.472240
900,JP,2025-12,NaN,NaN,NaN,0.005796,-0.013176,0.004976,NaN,0.043927,...,0.084299,0.002655,0.002863,NaN,NaN,NaN,0.264511,0.004569,0.041616,-4.175877
901,JP,2026-01,NaN,NaN,NaN,0.005291,-0.013183,0.004706,NaN,0.124617,...,0.176956,NaN,NaN,NaN,NaN,NaN,NaN,0.006569,0.062859,0.033746
902,JP,2026-02,NaN,NaN,NaN,NaN,NaN,-0.009937,NaN,0.079644,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.073583,-0.094860,2.526862


In [129]:
# Non-Stationarity Corrections - HP-filter cycle 
df = jp_aggregated_df.copy()
trans_df = trans_df.copy()

# Transformations: 
# - Output Trends: HP-filter cycle, trend smoothing
hpfilter_transformed_variables = ["Real GDP (billions chained 2015 JPY)"
                                 ]

# HP-filter cycle Transformed Variables 
for var in hpfilter_transformed_variables:    
    cycle, trend = hpfilter(
        np.log(df[f"{var}"]).dropna(),
        lamb=1600
    )
    trans_df[f"HPfilter-{var}"] = cycle
trans_df

,Country,Time,LogDiff-Monetary Aggregates - M1 (JPY),LogDiff-Monetary Aggregates - M2 (JPY),LogDiff-Monetary Aggregates - M3 (JPY),LogDiff-Total Treasury Reserves (- Gold),LogDiff-USD-JPY reer CPI-based (Index 2015=100),LogDiff-JPY-USD Spot Exchange Rate,LogDiff-HICP (SA),LogDiff-1615.T-Price,...,AR(1)detrend-Est. 1-year Neutral Interest Rate (%),AR(1)detrend-Est. 10-year Neutral Interest Rate (%),AR(1)detrend-Central Government Debt (% GDP),AR(1)detrend-Domestic Private Debt Securities (% GDP),AR(1)detrend-Domestic Public Debt Securities (% GDP),AR(1)detrend-Loan Interest Rate (%),AR(1)detrend-Deposit Interest Rate (%),AR(1)detrend-10-Year US T-Bills Yield (%),AR(1)detrend-CBOE-VIX,HPfilter-Real GDP (billions chained 2015 JPY)
0,JP,1950-12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,JP,1951-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,JP,1951-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,JP,1951-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,JP,1951-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
899,JP,2025-11,NaN,NaN,NaN,0.005714,-0.021022,0.024710,NaN,0.076439,...,0.002655,0.002863,NaN,NaN,NaN,-0.129205,0.004569,0.024252,1.472240,-0.001004
900,JP,2025-12,NaN,NaN,NaN,0.005796,-0.013176,0.004976,NaN,0.043927,...,0.002655,0.002863,NaN,NaN,NaN,0.264511,0.004569,0.041616,-4.175877,-0.001618
901,JP,2026-01,NaN,NaN,NaN,0.005291,-0.013183,0.004706,NaN,0.124617,...,NaN,NaN,NaN,NaN,NaN,NaN,0.006569,0.062859,0.033746,NaN
902,JP,2026-02,NaN,NaN,NaN,NaN,NaN,-0.009937,NaN,0.079644,...,NaN,NaN,NaN,NaN,NaN,NaN,0.073583,-0.094860,2.526862,NaN


In [130]:
# Transformed df to csv
jp_trans_df = trans_df.copy()
jp_trans_df.to_csv("Data/Transformed/jp_trans_df.csv", index=False)

### **5) TRANSFORMED VARIABLES STATIONARITY RE-TESTING**

In [131]:
# Autocorrelation coefficients AR(1)
# Drop non-numeric columns and rows with missing values
df = jp_trans_df.copy()
jp_transformed_numeric = df.drop(columns=["Country", "Time"]).dropna()

# AR(1) autocorrelation for each variable
ar1_results = {}
for col in jp_transformed_numeric.columns:
    series = jp_transformed_numeric[col]

# (!!!) lag-1 autocorrelation
    ar1 = series.autocorr(lag=1)
    ar1_results[col] = ar1

# Better to create a dataframe to display the results
jp_ar1_df = pd.DataFrame.from_dict(ar1_results, orient="index", columns=["AR(1)"])
jp_ar1_df

,AR(1)
LogDiff-Monetary Aggregates - M1 (JPY),0.484798
LogDiff-Monetary Aggregates - M2 (JPY),0.199956
LogDiff-Monetary Aggregates - M3 (JPY),0.321124
LogDiff-Total Treasury Reserves (- Gold),0.004811
LogDiff-USD-JPY reer CPI-based (Index 2015=100),0.373360
LogDiff-JPY-USD Spot Exchange Rate,0.343790
LogDiff-HICP (SA),0.291981
LogDiff-1615.T-Price,0.057838
LogDiff-BoJ’s Total Assets (100 Million Yen),-0.232157
AR(1)detrend-Total Credit - General Government (%GDP),-0.162293


In [132]:
# Unit-root Testing - Adfuller Test 
# Drop non-numeric columns and handle missing data
df = jp_trans_df.copy()
jp_trans_numeric = df.drop(columns=["Country", "Time"]).dropna()

# (!!!) We need to initialize the results as empty list before execuding the test
results = []

for col in jp_trans_numeric.columns:
    series = jp_trans_numeric[col]

# As before, we extract the AR(1) coefficients
    ar1 = series.autocorr(lag=1)

# Augmented Dickey-Fuller (ADF) unit root test 
    adf_result = adfuller(series, autolag="AIC")
    adf_stat = adf_result[0]
    p_value = adf_result[1]
    crit_values = adf_result[4]

    results.append({
        "Variable": col,
        "AR(1)": ar1,
        "ADF Statistic": adf_stat,
        "p-value": p_value,
        "Stationary - Absence of unit-root (HP1)": "Yes" if p_value < 0.05 else "No"
    })

jp_adf_df = pd.DataFrame(results)
jp_adf_df

,Variable,AR(1),ADF Statistic,p-value,Stationary - Absence of unit-root (HP1)
0,LogDiff-Monetary Aggregates - M1 (JPY),0.484798,-1.964948,3.021505e-01,No
1,LogDiff-Monetary Aggregates - M2 (JPY),0.199956,-8.334236,3.305323e-13,Yes
2,LogDiff-Monetary Aggregates - M3 (JPY),0.321124,-7.377829,8.628565e-11,Yes
3,LogDiff-Total Treasury Reserves (- Gold),0.004811,-4.579264,1.410270e-04,Yes
4,LogDiff-USD-JPY reer CPI-based (Index 2015=100),0.373360,-6.960905,9.170208e-10,Yes
5,LogDiff-JPY-USD Spot Exchange Rate,0.343790,-7.174666,2.747041e-10,Yes
6,LogDiff-HICP (SA),0.291981,-7.621960,2.116854e-11,Yes
7,LogDiff-1615.T-Price,0.057838,-9.755358,7.815983e-17,Yes
8,LogDiff-BoJ’s Total Assets (100 Million Yen),-0.232157,-1.405041,5.798571e-01,No
9,AR(1)detrend-Total Credit - General Government...,-0.162293,-3.383658,1.152875e-02,Yes


In [133]:
# Unit-root Testing - Phillips-Perron Test 
# (!!!) We need to initialize the results as empty list before execuding the test
pp_results = []

for col in jp_trans_numeric.columns:
    series = jp_trans_numeric[col].dropna()
    
# Phillips–Perron test 
# (!!!) From arch instead of stats.models is much smoother
    test = PhillipsPerron(series)
    pp_results.append({
        "Variable": col,
        "PP Statistic": test.stat,
        "p-value": test.pvalue,
        "Stationary - Absence of unit-root (HP1)": "Yes" if test.pvalue < 0.05 else "No"
    })

jp_pp_df = pd.DataFrame(pp_results)
jp_pp_df

,Variable,PP Statistic,p-value,Stationary - Absence of unit-root (HP1)
0,LogDiff-Monetary Aggregates - M1 (JPY),-7.259440,1.696692e-10,Yes
1,LogDiff-Monetary Aggregates - M2 (JPY),-8.550600,9.245770e-14,Yes
2,LogDiff-Monetary Aggregates - M3 (JPY),-7.738422,1.078027e-11,Yes
3,LogDiff-Total Treasury Reserves (- Gold),-10.295803,3.473932e-18,Yes
4,LogDiff-USD-JPY reer CPI-based (Index 2015=100),-6.671674,4.571222e-09,Yes
5,LogDiff-JPY-USD Spot Exchange Rate,-7.118697,3.771629e-10,Yes
6,LogDiff-HICP (SA),-7.798916,7.585275e-12,Yes
7,LogDiff-1615.T-Price,-9.743961,8.351410e-17,Yes
8,LogDiff-BoJ’s Total Assets (100 Million Yen),-12.755525,8.355802e-24,Yes
9,AR(1)detrend-Total Credit - General Government...,-12.377530,5.115567e-23,Yes


### **6) DESCRIPTIVE STATISTICS (TRANSFORMED)**

In [134]:
# Descriptive statistics summary table - Raw Data
df = jp_aggregated_df[["Country", "Time", "HICP (SA)", "Call Money/Interbank Immediate (%)", "10-Year Gov Bond Yields (%)"]].copy()
df.describe()

,HICP (SA),Call Money/Interbank Immediate (%),10-Year Gov Bond Yields (%)
count,664.000000,487.000000,445.000000
mean,85.737008,1.247216,1.810333
std,19.358703,2.185570,1.825189
min,30.114560,-0.071000,-0.280000
25%,80.909821,0.001000,0.566000
50%,95.116012,0.091000,1.333000
75%,97.629878,0.804870,1.948000
max,111.553413,8.278130,8.032000


In [135]:
# Descriptive statistics summary table - Transformed Data
jp_core_trans_df = jp_trans_df[["Country", "Time", "LogDiff-HICP (SA)", "AR(1)detrend-Call Money/Interbank Immediate (%)", "AR(1)detrend-10-Year Gov Bond Yields (%)"]].copy()
df = jp_core_trans_df.copy()
df.to_csv("Data/Transformed/jp_core_trans_df.csv", index=False)
df.describe()

,LogDiff-HICP (SA),AR(1)detrend-Call Money/Interbank Immediate (%),AR(1)detrend-10-Year Gov Bond Yields (%)
count,663.000000,4.860000e+02,4.440000e+02
mean,2.370126,-2.160595e-15,-8.951798e-17
std,6.175389,1.569123e-01,1.572884e-01
min,-23.983093,-1.127125e+00,-5.849537e-01
25%,-0.707249,-5.201793e-03,-7.059929e-02
50%,1.129541,1.905528e-03,-6.742910e-03
75%,4.199134,1.192572e-02,6.081615e-02
max,50.654876,8.870219e-01,9.596040e-01
